# Live bar sanity check (HIMS)

This notebook fetches recent bars using the same Alpaca broker client as the API and prints the newest bar timestamp.

Prereqs:
- `ALPACA_API_KEY` and `ALPACA_SECRET_KEY` are set in the environment.
- Repo exists at `/home/tglim/codes/AutoBuySell`.


In [1]:
import sys
from datetime import datetime, timezone
import pandas as pd

sys.path.append("/home/tglim/codes/AutoBuySell/api")

from app.brokers.alpaca import AlpacaBroker

symbol = "HIMS"
timeframe = "30Min"
limit = 40

broker = AlpacaBroker()
bars = await broker.get_historicals(symbol, timeframe, limit)

len(bars)


/tmp/ipykernel_1791176/304035591.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


40

In [2]:
if not bars:
    print("No bars returned")
else:
    print("Raw first timestamp:", bars[0].timestamp)
    print("Raw last timestamp:", bars[-1].timestamp)
    rows = [
        {
            "timestamp": b.timestamp,
            "open": b.open,
            "high": b.high,
            "low": b.low,
            "close": b.close,
            "volume": b.volume,
        }
        for b in bars
    ]
    df = pd.DataFrame(rows)
    df_sorted = df.sort_values("timestamp")
    display(df_sorted.tail(10))
    newest = df_sorted["timestamp"].iloc[-1]
    now = datetime.now(timezone.utc)
    print("Newest bar:", newest)
    print("Now (UTC):", now)
    print("Bar age:", now - newest)


Raw first timestamp: 2026-01-20 13:00:00+00:00
Raw last timestamp: 2026-01-22 19:00:00+00:00


,timestamp,open,high,low,close,volume
30,2026-01-22 14:30:00+00:00,29.405,29.560,29.000,29.300,55578.0
31,2026-01-22 15:00:00+00:00,29.265,29.460,29.160,29.360,36393.0
32,2026-01-22 15:30:00+00:00,29.340,29.540,29.340,29.440,70906.0
33,2026-01-22 16:00:00+00:00,29.450,29.515,29.255,29.300,37530.0
34,2026-01-22 16:30:00+00:00,29.315,29.390,29.210,29.300,17925.0
35,2026-01-22 17:00:00+00:00,29.320,29.440,29.285,29.435,38067.0
36,2026-01-22 17:30:00+00:00,29.450,29.565,29.440,29.555,26034.0
37,2026-01-22 18:00:00+00:00,29.540,29.810,29.480,29.740,38683.0
38,2026-01-22 18:30:00+00:00,29.740,29.850,29.730,29.840,30051.0
39,2026-01-22 19:00:00+00:00,29.870,30.370,29.825,30.370,42811.0


Newest bar: 2026-01-22 19:00:00+00:00
Now (UTC): 2026-01-25 01:38:26.489416+00:00
Bar age: 2 days 06:38:26.489416


In [5]:
from app.domain.models import Candle
from app.strategies.mean_reversion import MeanReversionStrategy
from app.strategies.base import StrategyContext
from app.brokers.base import AccountInfo

if not bars:
    print("No bars to evaluate")
else:
    candles = [
        Candle(
            symbol=symbol,
            timeframe=timeframe,
            timestamp=b.timestamp,
            open=b.open,
            high=b.high,
            low=b.low,
            close=b.close,
            volume=b.volume,
        )
        for b in bars
    ]

    strategy = MeanReversionStrategy()
    await strategy.initialize({"timeframe": timeframe})
    account = AccountInfo(
        account_id="TEST",
        currency="USD",
        cash=10000.0,
        portfolio_value=10000.0,
        buying_power=10000.0,
        is_paper=True,
    )
    context = StrategyContext(symbol=symbol, account=account, params={})
    signals = await strategy.on_bar(context, candles)
    print(signals)


[]
